# Login via browser

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import InteractiveBrowserCredential
# ==========================================
# 1. Configuration & Job Mapping
# ==========================================
TENANT_ID       = "ebbb4fc3-587b-477c-a1e8-ee47d8c02546"
SUBSCRIPTION_ID = "ab211f7b-463f-4833-9605-d260e596a35a"
RESOURCE_GROUP  = "73da10b4-5dff-54e2-db0d-3a1fab882485"
WORKSPACE_NAME  = "73da10b45dff54e2db0d3a1fab882485"


# ==========================================
# 2. Authentication & Client Setup
# ==========================================
print("Authenticating...")
credential = InteractiveBrowserCredential(tenant_id=TENANT_ID)

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

# Load into RAM and save to a local folder a single file

In [7]:
import fsspec
from pathlib import Path
import nibabel as nib
import gzip
from azure.identity import InteractiveBrowserCredential

# ==========================================
# 1. Azure ML Configuration
# ==========================================
SUBSCRIPTION_ID = "ab211f7b-463f-4833-9605-d260e596a35a"
RESOURCE_GROUP  = "73da10b4-5dff-54e2-db0d-3a1fab882485"
WORKSPACE_NAME  = "73da10b45dff54e2db0d3a1fab882485"
DATASTORE_NAME  = "73da10b45dff54e2db0d3a1fab882485"

# The specific scan you want to load right now
DATA_DIR = "NIFTI/DATA_REGISTERED_FIXED_LATE/"
PID = "TAVI_002"

# Where to save it on your computer
LOCAL_SAVE_PATH = f"./output/{PID}_CT_ANGIO.nii.gz"
Path(LOCAL_SAVE_PATH).parent.mkdir(exist_ok=True, parents=True)

# Automatically construct the fsspec URI
uri = f"azureml://subscriptions/{SUBSCRIPTION_ID}/resourcegroups/{RESOURCE_GROUP}/workspaces/{WORKSPACE_NAME}/datastores/{DATASTORE_NAME}/paths/{DATA_DIR}/{PID}/CT_LATE.nii.gz"


# ==========================================
# 2. Streaming
# ==========================================
print(f"Streaming {PID}/{uri.split('/')[-1]} directly into RAM...")

with fsspec.open(uri, "rb", credential=credential) as remote_file:
    # 1. READ ONCE: Pull all bytes from Azure into your Mac's RAM
    compressed_bytes = remote_file.read()

# 2. SAVE TO DISK: Write those bytes to a file for 3D Slicer
print(f"Saving compressed file to: {LOCAL_SAVE_PATH}")
with open(LOCAL_SAVE_PATH, "wb") as local_file:
    local_file.write(compressed_bytes)

# ==========================================
# 3. Decompression & Loading
# ==========================================
print("Decompressing data...")
uncompressed_bytes = gzip.decompress(compressed_bytes)

# Load into nibabel without saving to disk
nifti_img = nib.Nifti1Image.from_bytes(uncompressed_bytes) 
data = nifti_img.get_fdata()

print("--------------------------------------")
print("✅ Success! Data loaded into memory.")
print(f"Image shape: {data.shape}")
print("--------------------------------------")

Streaming TAVI_002/CT_LATE.nii.gz directly into RAM...
Saving compressed file to: ./output/TAVI_002_CT_ANGIO.nii.gz
Decompressing data...
--------------------------------------
✅ Success! Data loaded into memory.
Image shape: (512, 512, 57)
--------------------------------------


# Download model weights after successful training

In [ ]:
JOB_NAME = "amusing_glass_6wtpqqn5t1"
LOCAL_DOWNLOAD_DIR = "./output"

# ==========================================
# Download the Exact YAML Output
# ==========================================
print(f"Downloading model weights from job: {JOB_NAME}...")

ml_client.jobs.download(
    name=JOB_NAME, 
    download_path=LOCAL_DOWNLOAD_DIR, 
    output_name="output_model" 
)

print("--------------------------------------")
print(f"✅ Download complete!")
print("--------------------------------------")

Opening browser for authentication (Targeting exact Tenant)...


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


/Volumes/X9Pro/HI/azure/lib/python3.10/site-packages/msal/oauth2cli/oauth2.py:408: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(
Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy 'https://stgsramlprdwevzhst6.blob.core.windows.net/azureml-blobstore-67b57835-3de4-400e-89c3-68cb536ca3ec/azureml/amusing_glass_6wtpqqn5t1/output_model/' 'output/named-outputs/output_model' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.


--------------------------------------
✅ Download complete!
--------------------------------------


# Retrieve latest job's name within a specific experiment

In [9]:
experiment_name = "myocardium_val_full_cv"

print(f"Searching for the latest job in: {experiment_name}...")

# 1. Get the list without the buggy keyword
all_jobs = ml_client.jobs.list()

# 2. Manually find the first job that matches your experiment
latest_job = None
for job in all_jobs:
    # Azure returns jobs in descending order (newest first)
    if job.experiment_name == experiment_name:
        latest_job = job
        break

if latest_job:
    JOB_NAME = latest_job.name
    print(f"✅ Automatically detected: {JOB_NAME}")
    print(f"Status: {latest_job.status}")
else:
    print(f"❌ No jobs found for experiment '{experiment_name}'")

Searching for the latest job in: myocardium_val_full_cv...
✅ Automatically detected: teal_feather_2n1ny84j8h
Status: Completed


# Download a specific job's output into a local folder

In [10]:
ml_client.jobs.download(
    name=JOB_NAME, 
    download_path="./cv_inference_results", 
    output_name="validation_results"
)

print("Download complete. Files are located in the './cv_inference_results' directory.")

Download complete. Files are located in the './cv_inference_results' directory.
